In [1]:
import io
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import requests
from girder_client import GirderClient


def get_girder_client(session=None):
    gc = GirderClient(
        apiUrl=os.environ.get("GIRDER_API_URL", "https://girder.htmdec.org/api/v1")
    )
    gc._session = session
    try:
        gc.authenticate(apiKey=os.environ["GIRDER_API_KEY"])
    except KeyError:
        raise RuntimeError("GIRDER_API_KEY environment variable not set")

    me = gc.get("user/me")
    assert me is not None, "Failed to authenticate with Girder API"
    return gc


def get_item_to_memory(gc, item):
    """Download a Girder item into memory."""
    item_id = item["_id"]
    response = gc.sendRestRequest(
        "GET", f"item/{item_id}/download", stream=True, jsonResp=False
    )
    response.raise_for_status()
    for chunk in response.iter_content(chunk_size=65536):
        yield chunk


def fetch_and_parse(gc, item):
    """Download a Girder item and parse its results."""
    content = io.BytesIO(b"".join(get_item_to_memory(gc, item)))
    return parse_results(content, item)


def parse_results(content, item):
    data = {
        k: v.strip()
        for k, v in (line.decode("utf8").split(",") for line in content.readlines())
    }
    data["Date"] = data["Date"] + " " + data.pop("Time")
    data["igsn"] = item["meta"]["igsn"]
    data["itemId"] = item["_id"]
    return data


def coerce_types(df):
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Run Time"] = pd.to_timedelta(df["Run Time"], errors="coerce")
    numeric_columns = [
        "Velocity at Max Compression",
        "Time at Max Compression",
        "Velocity at Max Tension",
        "Time at Max Tension",
        "Velocity at Recompression",
        "Time at Recompression",
        "Carrier Frequency",
        "Spall Strength",
        "Spall Strength Uncertainty",
        "Strain Rate",
        "Strain Rate Uncertainty",
        "Peak Shock Stress",
        "Spect Time Res",
        "Spect Freq Res",
        "Spect Velocity Res",
        "Signal Start Time",
        "Smoothing Characteristic Time",
        "HEL Strength (GPa)",
        "HEL Uncertainty (GPa)",
        "HEL Free Surface Velocity (m/s)",
        "HEL Time Detection (ns)",
        "HEL Consecutive Points",
        "HEL Segment Duration (ns)",
        "HEL Strain Rate",
    ]
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in ["Velocity OK", "Spall OK", "Uncertainty OK", "HEL Detected"]:
        if col in df.columns:
            df[col] = df[col].map({"True": True, "False": False})

    return df

In [2]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    rows = []
    limit = 50
    offset = 0
    with ThreadPoolExecutor(max_workers=8) as executor:
        while True:
            batch = gc.get(
                "aimdl/datafiles",
                parameters={
                    "limit": limit,
                    "offset": offset,
                    "dataType": "pdv_alpss_results",
                },
            )
            if not batch:
                break
            futures = [executor.submit(fetch_and_parse, gc, item) for item in batch]
            for future in as_completed(futures):
                rows.append(future.result())
            if len(batch) < limit:
                break
            offset += limit

    df = pd.DataFrame(rows)
    df = coerce_types(df)

/tmp/ipykernel_16100/3324275922.py:54: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [3]:
df.head()

,Date,File Name,Velocity OK,Spall OK,Uncertainty OK,Error Message,Run Time,Velocity at Max Compression,Time at Max Compression,Velocity at Max Tension,...,HEL Detected,HEL Strength (GPa),HEL Uncertainty (GPa),HEL Free Surface Velocity (m/s),HEL Time Detection (ns),HEL Consecutive Points,HEL Segment Duration (ns),HEL Strain Rate,igsn,itemId
0,2026-03-17 21:18:00,C1--20251022--00001.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.865158,772.344404,0.000001,91.429072,...,False,NaN,NaN,-1.798127,992.968750,4,0.023437,NaN,JHAMAB00021-16,69b9c5197590bb7e1ed22400
1,2026-03-17 21:17:00,C1--20251022--00009.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.848154,1092.321038,0.000001,-123.587673,...,False,NaN,NaN,-3.337541,980.039062,101,0.781250,NaN,JHAMAB00021-16,69b9c4ea4e69e623f00204be
2,2026-03-17 21:17:00,C1--20251022--00010.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.846242,1094.988503,0.000001,124.939287,...,False,NaN,NaN,-8.952012,1063.843750,10,0.070313,NaN,JHAMAB00021-16,69b9c4e54d9c815b961aa2cd
3,2026-03-17 21:17:00,C1--20251022--00011.csv,True,True,True,,0 days 00:00:00.940500,1052.412379,0.000001,179.923642,...,True,0.249045,0.041728,12.452236,961.578125,9,0.062500,0.000253,JHAMAB00021-16,69b9c4df32b20c7a63167dc7
4,2026-03-17 21:17:00,C1--20251022--00012.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.839444,1112.837979,0.000001,167.214952,...,False,NaN,NaN,3.510935,984.585937,61,0.468750,NaN,JHAMAB00021-16,69b9c4d94e69e623f00204a4
